<a href="https://colab.research.google.com/github/sergioGarcia91/SeismicUP/blob/main/01_EDA_inicial_SGC.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Este cuaderno integra y normaliza los catálogos sísmicos descargados del Servicio Geológico Colombiano (SGC - https://bdrsnc.sgc.gov.co/paginas1/catalogo) y realiza un análisis preliminar de la distribución espacial de los eventos en el área del Campo Colorado. Se establece un punto central de referencia y, a partir de él, se examina la ubicación relativa de los sismos para identificar patrones de concentración y posibles vacíos de registro.

# Inicio

In [ ]:
!git clone https://github.com/sergioGarcia91/SeismicUP.git

In [ ]:
# para incluir la libreria clonada
import sys
sys.path.append("/content/SeismicUP")

import seismicup as sup

In [ ]:
!pip -q install python-calamine

In [ ]:
!pip3 install contextily

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import geopandas as gpd
import seaborn as sns
import os
import re
import contextily as cx #para el basemap en geopandas
import xyzservices.providers as xyz #para escoger el basemap
import seismicup as sup
import urllib.request
import matplotlib.font_manager as fm

from matplotlib.colors import LogNorm # para la escala logaritmica de los colores
from matplotlib.ticker import LogFormatter, LogLocator # escala log
from scipy import stats # regresion lineal

# Conectar al Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Cambiar Fuente

In [ ]:
sup.plots.get_TimesNewRoman_font()

# Paths

In [ ]:
path_save_figures = '/content/drive/MyDrive/Contratos/20250801_UIS_CO2/Notebooks/Figuras_01_EDA'

In [ ]:
path_shape_municipios = '/content/drive/MyDrive/Contratos/20250801_UIS_CO2/SIG/MGN2020_MPIO_POLITICO'

In [ ]:
path_datasets = '/content/SeismicUP/Datasets_'

os.listdir(path_datasets)

# Cargar catalogos

In [ ]:
lista_catalogos = ['Cat_sis_SGC_jun1993_feb2018',
                   'Cat_sis_SGC_mar2018_2024',
                   'Cat_sis_SGC_TECTO_feb2014_2024',
                   'Cat_sis_SGC_LBG_jun1993_2021']

for i in lista_catalogos:
  cantidad_archivos = len(os.listdir(os.path.join(path_datasets, i)))
  print(f'{i} = ', cantidad_archivos)

In [ ]:
df_general = pd.DataFrame(columns=['Fecha-Hora UTC',
                                   'Latitud', # grados
                                   'Longitud', # grados
                                   'Profundidad [km]',
                                   'Magnitud',
                                   'Tipo Magnitud',
                                   'Error Latitud [km]',
                                   'Error Longitud [km]',
                                   'Error Profundidad [km]',
                                   'Numero de Fases',
                                   'RMS [seg]',
                                   'Gap', # grados
                                   'Departamento',
                                   'Municipio',
                                   ])

df_general.head()

## Catalogo SGC 1

In [ ]:
lista_catalogos

In [ ]:
path_catalogo = os.path.join(path_datasets, 'Cat_sis_SGC_jun1993_feb2018')
os.listdir(path_catalogo)

df_SGC_1 = df_general.copy()

for i in os.listdir(path_catalogo)[:]:
  if i.endswith('.xlsx'):
    print('-----'*3)
    print('File: ', i)
    print('-----'*3)
    df_temp_1 = sup.io.crear_catalogo_SGC_1(path_catalogo=path_catalogo,
                                            file_excel=i)

    df_SGC_1 = pd.concat([df_SGC_1, df_temp_1], ignore_index=True)
    df_SGC_1.reset_index(drop=True, inplace=True)

    del df_temp_1


In [ ]:
df_SGC_1['Catalogo'] = 'SGC 1'
df_SGC_1.info()

In [ ]:
df_SGC_1.head()

## Catalogo SGC 2

In [ ]:
lista_catalogos

In [ ]:
path_catalogo = os.path.join(path_datasets, 'Cat_sis_SGC_mar2018_2024')
os.listdir(path_catalogo)

df_SGC_2 = df_general.copy()

for i in os.listdir(path_catalogo)[:]:
  if i.endswith('.xlsx'):
    print('-----'*3)
    print('File: ', i)
    print('-----'*3)
    df_temp_1 = sup.io.crear_catalogo_SGC_2(path_catalogo=path_catalogo,
                                            file_excel=i)

    df_SGC_2 = pd.concat([df_SGC_2, df_temp_1], ignore_index=True)
    df_SGC_2.reset_index(drop=True, inplace=True)

    del df_temp_1


In [ ]:
df_SGC_2['Catalogo'] = 'SGC 2'
df_SGC_2.info()

In [ ]:
df_SGC_2.head()

## Catalogo TECTO

In [ ]:
lista_catalogos

In [ ]:
path_catalogo = os.path.join(path_datasets, 'Cat_sis_SGC_TECTO_feb2014_2024')
os.listdir(path_catalogo)

df_SGC_TECTO = df_general.copy()

for i in os.listdir(path_catalogo)[:]:
  if i.endswith('.xlsx') and (i != 'excel_estaciones.xlsx'):
    print('-----'*3)
    print('File: ', i)
    print('-----'*3)
    df_temp_1 = sup.io.crear_catalogo_SGC_TECTO(path_catalogo=path_catalogo,
                                                file_excel=i)

    print(df_temp_1.shape)
    df_SGC_TECTO = pd.concat([df_SGC_TECTO, df_temp_1], ignore_index=True)
    df_SGC_TECTO.reset_index(drop=True, inplace=True)

    del df_temp_1


In [ ]:
df_SGC_TECTO['Catalogo'] = 'TECTO'
df_SGC_TECTO['Profundidad [km]'] = df_SGC_TECTO['Profundidad [km]'].astype(float)

df_SGC_TECTO.info()

In [ ]:
df_SGC_TECTO.head()

## Catalogo LBG

In [ ]:
lista_catalogos

In [ ]:
path_catalogo = os.path.join(path_datasets, 'Cat_sis_SGC_LBG_jun1993_2021')
os.listdir(path_catalogo)

df_SGC_LBG = df_general.copy()

for i in os.listdir(path_catalogo)[:]:
  if i.endswith('.xlsx'):
    print('-----'*3)
    print('File: ', i)
    print('-----'*3)
    df_temp_1 = sup.io.crear_catalogo_SGC_LBG(path_catalogo=path_catalogo,
                                              file_excel=i)

    print(df_temp_1.shape)
    df_SGC_LBG = pd.concat([df_SGC_LBG, df_temp_1], ignore_index=True)
    df_SGC_LBG.reset_index(drop=True, inplace=True)

    del df_temp_1


In [ ]:
df_SGC_LBG['Catalogo'] = 'LBG'
df_SGC_LBG.info()

In [ ]:
df_SGC_LBG.head()

# Unir catalogos

In [ ]:
df_unido = pd.concat([df_SGC_1, df_SGC_2, df_SGC_TECTO, df_SGC_LBG], ignore_index=True)
df_unido.reset_index(drop=True, inplace=True)

df_unido.head()

In [ ]:
df_unido.info()

## Cargar Shape municipios

In [ ]:
shape_municipios = gpd.read_file(os.path.join(path_shape_municipios, 'MGN_MPIO_POLITICO.shp'))
shape_municipios = shape_municipios.to_crs(epsg=9377)


shape_municipios.boundary.plot()

# Geodataframe

El DataFrame se convertirá en un GeoDataFrame para reproyectar las coordenadas. Luego, las coordenadas transformadas se devolverán al DataFrame “plano” para continuar con el análisis exploratorio (EDA).

In [ ]:
gdf_unido = gpd.GeoDataFrame(
    df_unido, geometry=gpd.points_from_xy(df_unido.Longitud, df_unido.Latitud), crs="EPSG:4326"
)

In [ ]:
gdf_unido.plot()

In [ ]:
gdf_unido = gdf_unido.to_crs(9377)

gdf_unido.plot()

## Obtener X y Y

In [ ]:
gdf_unido.get_coordinates()

In [ ]:
df_unido['X [m]'] = gdf_unido.get_coordinates()['x']
df_unido['Y [m]'] = gdf_unido.get_coordinates()['y']

df_unido.head()

In [ ]:
df_unido.info()

# DataFrame Unido

In [ ]:
df_unido['Tiempo'] = pd.to_datetime(df_unido['Fecha-Hora UTC'])

df_unido.info()

In [ ]:
df_unido.describe().round(2)

In [ ]:
df_unido['Tipo Magnitud'].unique()

El catalogo unido presenta diversidad de tipso de magnitudes. Se debe considerar una priorizacion de magntidudes para trabajar y realizar el análisis.

In [ ]:
total_duplicados = df_unido.duplicated(subset=['Fecha-Hora UTC', 'Latitud', 'Longitud'], keep=False).sum()
# se tiene 42396 duplicados, se debe escoger cual quedarse ...
total_eventos_df_unido = len(df_unido)
diferencia_eventos = total_eventos_df_unido - total_duplicados
porcentaje_duplicados = (total_duplicados / total_eventos_df_unido) * 100

print('Total eventos: ', total_eventos_df_unido)
print('Total duplicados: ', total_duplicados)
print('Diferencia: ', diferencia_eventos)
print('Porcentaje de duplicados: ', round(porcentaje_duplicados, 2), '%')


Menos del 14% de los eventos del catálogo son duplicados; por tanto, pueden eliminarse sin afectar significativamente el análisis.

In [ ]:
df_duplicados = df_unido[df_unido.duplicated(subset=['Fecha-Hora UTC', 'Latitud', 'Longitud'], keep=False)].sort_values(by=['Fecha-Hora UTC', 'Latitud', 'Longitud']).copy()
df_duplicados.reset_index(drop=True, inplace=True)

df_duplicados

El catálogo SGC-2 parece más completo que TECTO: por ejemplo, incluye la columna FASE, ausente en TECTO. Además, se observan discrepancias puntuales en los valores de magnitud entre ambas fuentes.

In [ ]:
df_duplicados['Tipo Magnitud'].unique()

In [ ]:
df_unido.columns

In [ ]:
df_unido.loc[:, 'Catalogo'].value_counts().sort_values(ascending=False)

In [ ]:
df_unido.loc[:, 'Catalogo'].value_counts().sort_values(ascending=False).sum()

Los catálogos SGC-1 y SGC-2 registran más eventos, probablemente porque incluyen parte de la Mesa de Los Santos y poseen mayor cobertura (espacial y/o en profundidad hipocentral).

In [ ]:
df_unido['Tiempo'].dt.date.min()

In [ ]:
for catalogo_ in df_unido.loc[:, 'Catalogo'].unique():
  print('--------'*3)
  print('--------'*3)
  print('Catalogo: ', catalogo_)
  df_temp = df_unido[df_unido['Catalogo'] == catalogo_].copy()
  df_temp.reset_index(drop=True, inplace=True)

  fecha_minima = df_temp['Tiempo'].dt.date.min()
  fecha_maxima = df_temp['Tiempo'].dt.date.max()
  total_dias = (fecha_maxima - fecha_minima).days
  total_eventos = len(df_temp)
  total_duplicados = df_temp.duplicated(subset=['Fecha-Hora UTC', 'Latitud', 'Longitud'], keep=False).sum()
  porcentaje_duplicados = (total_duplicados / total_eventos) * 100
  prof_minima = df_temp['Profundidad [km]'].min()
  prof_maxima = df_temp['Profundidad [km]'].max()
  tipos_magnitudes = df_temp['Tipo Magnitud'].unique()
  mag_minima = df_temp['Magnitud'].min()
  mag_maxima = df_temp['Magnitud'].max()

  print('Fecha mínima: ', fecha_minima)
  print('Fecha máxima: ', fecha_maxima)
  print('Total días: ', total_dias)
  print('Total eventos: ', total_eventos)
  print('Total duplicados: ', total_duplicados)
  print('Porcentaje de duplicados: ', round(porcentaje_duplicados, 2), '%')
  print('Profundidad mínima [km]: ', prof_minima)
  print('Profundidad máxima [km]: ', prof_maxima)
  print('Tipos de magnitudes: ', tipos_magnitudes)
  print('Magnitud mínima: ', mag_minima)
  print('Magnitud máxima: ', mag_maxima)

  del df_temp
  print('\n')

# DataFrame Filtrado

Considerando la presencia de duplciados y que no todo el catalogo está compelto, se va proceder a eliminar los duplicados manteniendo solo el proimer evento. Esto considerando que los Catalogos SGC 1 y 2 al parecer son los mas completos, aunque presentan duplicados respecto a los catalogos de TECTO y LBG, de ser requedrido luego se procederá a ajustar el `df_filtrado`.

In [ ]:
df_filtrado = df_unido.copy()
df_filtrado.drop_duplicates(subset=['Fecha-Hora UTC', 'Latitud', 'Longitud'], keep='first', inplace=True)
df_filtrado.reset_index(drop=True, inplace=True)

df_filtrado.head()

In [ ]:
df_filtrado.info()

In [ ]:
df_filtrado[df_filtrado['Error Latitud [km]'].isna()]

In [ ]:
df_filtrado[df_filtrado['Error Latitud [km]'].isna()].loc[:, 'Catalogo'].value_counts()

Algunos eventos no reportan errores de localización y, por su formato, parecen provenir del catálogo SGC-1. Dado que son pocos y cuentan con coordenadas válidas, se conservarán en el análisis.

In [ ]:
df_filtrado.describe().round(2)

In [ ]:
df_filtrado['X [km]'] = df_filtrado['X [m]'] / 1000
df_filtrado['Y [km]'] = df_filtrado['Y [m]'] / 1000

In [ ]:
df_filtrado.columns

### Histograma errores

In [ ]:
fig, ax = plt.subplots(nrows=2, ncols=3, figsize=(12,8))

sup.plots.histograma_plot(df= df_filtrado,
                          columna= 'Y [km]',
                          ax_h= ax[0,0],
                          dist_min= df_filtrado['Y [km]'].min(),
                          step_= 20)

sup.plots.histograma_plot(df= df_filtrado,
                          columna= 'X [km]',
                          ax_h= ax[0,1],
                          dist_min= df_filtrado['X [km]'].min(),
                          step_= 20)

sup.plots.histograma_plot(df= df_filtrado,
                          columna= 'Profundidad [km]',
                          ax_h= ax[0,2],
                          dist_min= df_filtrado['Profundidad [km]'].min(),
                          step_= 20)

sup.plots.histograma_plot(df= df_filtrado,
                          columna= 'Error Latitud [km]',
                          ax_h= ax[1,0],
                          dist_min= 0,
                          step_= 10)

sup.plots.histograma_plot(df= df_filtrado,
                          columna= 'Error Longitud [km]',
                          ax_h= ax[1,1],
                          dist_min= 0,
                          step_= 10)

sup.plots.histograma_plot(df= df_filtrado,
                          columna= 'Error Profundidad [km]',
                          ax_h= ax[1,2],
                          dist_min= 0,
                          step_= 20)

ax[0,2].set_xlabel('Depth [km]')
ax[1,0].set_xlabel('Latitude Error [km]')
ax[1,1].set_xlabel('Longitude Error [km]')
ax[1,2].set_xlabel('Depth Error [km]')

sup.plots.save_plot(path_save_figuras = path_save_figures,
                    file_name = 'hitograma_errores_df_filtrado',
                    formato = 'png',
                    dpi_ = 300)

plt.show()

### Tiempo vs magnitud

In [ ]:
df_filtrado.columns

In [ ]:
fig, ax = plt.subplots(ncols=1, nrows=2, figsize=(10,10))

filtro_datos = df_filtrado['Profundidad [km]'] <= 50

sup.plots.time_magnitud_plot(df = df_filtrado,
                             filtro_datos = filtro_datos,
                             columna_eje_x_tiempo = 'Tiempo',
                             comlumna_eje_y = 'Magnitud',
                             ax_s = ax[0],
                             label_s= f'{filtro_datos.sum()} Earthquakes [Depth $\leq50$ km]')
ax[0].legend()
ax[0].set_xlabel('Time')
ax[0].set_ylabel('Magnitude')
#ax[0].set_title()
#plt.ight_layout()

filtro_datos = df_filtrado['Profundidad [km]'] > 50

sup.plots.time_magnitud_plot(df = df_filtrado,
                             filtro_datos = filtro_datos,
                             columna_eje_x_tiempo = 'Tiempo',
                             comlumna_eje_y = 'Magnitud',
                             ax_s = ax[1],
                             label_s= f'{filtro_datos.sum()} Earthquakes [Depth $>50$ km]')
ax[1].legend()
ax[1].set_xlabel('Time')
ax[1].set_ylabel('Magnitude')
#ax[0].set_title()

sup.plots.save_plot(path_save_figuras = path_save_figures,
                    file_name = 'scatter_tiempo_magnitud',
                    formato = 'png',
                    dpi_ = 300)

plt.show()

### Corte

In [ ]:
df_filtrado.columns

In [ ]:
fig, ax = plt.subplots(ncols=1, nrows=2, figsize=(8,8))

filtro_datos = df_filtrado['Profundidad [km]'] > -10000

sup.plots.plot_corte(df = df_filtrado,
                     columna_x = 'X [km]',
                     columna_y = 'Profundidad [km]',
                     filtro_datos = filtro_datos,
                     ax_s = ax[0],
                     label_s = 'Vista Oeste-Este',
                     invert_yaxis=True)
ax[0].set_title('West–East View')

filtro_datos = df_filtrado['Profundidad [km]'] > -10000

sup.plots.plot_corte(df = df_filtrado,
                     columna_x = 'Y [km]',
                     columna_y = 'Profundidad [km]',
                     filtro_datos = filtro_datos,
                     ax_s = ax[1],
                     label_s = 'Vista Oeste-Este',
                     invert_yaxis=True)
ax[1].set_title('South–North View')

ax[0].set_ylabel('Depth [km]')
ax[1].set_ylabel('Depth [km]')

plt.tight_layout()

sup.plots.save_plot(path_save_figuras = path_save_figures,
                    file_name = 'cortes_WE_NS',
                    formato = 'png',
                    dpi_ = 300)

plt.show()

# Punto referencia Campo Colorado



In [ ]:
punto_xy_CampoColorado = (4919222.973, 2308553.476) # x, y en metros

# se va calcular la distancia de los eventos al centroide o punto de referencia
# del Campo Colorado solo en XY mas no en profundidad

df_filtrado['Distancia [m]'] = np.sqrt((df_filtrado['X [m]'] - punto_xy_CampoColorado[0])**2 + (df_filtrado['Y [m]'] - punto_xy_CampoColorado[1])**2)
df_filtrado['Distancia [km]'] = df_filtrado['Distancia [m]'] / 1000

df_filtrado.describe().round(2)

In [ ]:
print('Distancia mínima [km]: ', df_filtrado['Distancia [km]'].min())
print('Distancia mínima [km]: ', df_filtrado['Distancia [km]'].max())

In [ ]:
df_filtrado['Distancia [km]'].min()

### Histograma 2D - Mapa

In [ ]:
x_min = df_filtrado['X [m]'].min()
x_max = df_filtrado['X [m]'].max()
y_min = df_filtrado['Y [m]'].min()
y_max = df_filtrado['Y [m]'].max()

steps_ = 5 * 1000 # m
bins_x = np.arange(start=int(x_min)-steps_, stop=int(x_max)+steps_, step=steps_)
bins_y = np.arange(start=int(y_min)-steps_, stop=int(y_max)+steps_, step=steps_)

# Datos y bins (usa los tuyos: bins_x, bins_y)
x = df_filtrado['X [m]'].to_numpy()
y = df_filtrado['Y [m]'].to_numpy()

x_min, x_max = bins_x[0], bins_x[-1]
y_min, y_max = bins_y[0], bins_y[-1]

fig, ax = plt.subplots(figsize=(7,9))

# hist2d con conteos en log (evita log(0) con vmin/cmin=1)
H = ax.hist2d(
    x, y,
    bins=[bins_x, bins_y],
    cmap='plasma', # rampa naranja
    norm=LogNorm(vmin=1), # escala log (base indiferente; la etiqueta la ponemos base-10)
    cmin=1, # descarta bins con 0 para evitar log(0)
    shading='auto',
    alpha= 0.7
)

# Barra de color en LOG10 (ticks en potencias de 10)
cb = fig.colorbar(H[3], ax=ax, pad=0.01)
cb.locator = LogLocator(base=10) # ticks en 1, 10, 100, ...
cb.formatter = LogFormatter(base=10, labelOnlyBase=False)
cb.update_ticks()
cb.set_label('Events [log10]')

# Limites y overlays (usa los tuyos)
ax.set_xlim(bins_x[0], bins_x[-1])
ax.set_ylim(bins_y[0], bins_y[-1])

# Bordes de municipios encima del histograma
shape_municipios.boundary.plot(ax=ax,
                               edgecolor='black',
                               lw=1)

ax.scatter(x=punto_xy_CampoColorado[0],
            y=punto_xy_CampoColorado[1],
            label='Campo Colorado',
            c='r',
            s=20)

ax.legend()

# Mapa base debajo (ajusta zorder)
cx.add_basemap(ax=ax,
               crs='epsg:9377',
               source=xyz.OpenTopoMap,
               reset_extent=True,
               attribution_size=2,
               zorder=0)

ax.set_xlabel('X [m]')
ax.set_ylabel('Y [m]')
#ax.set_title('Histograma 2D (escala log10)')
ax.grid(ls='--', c='k', alpha=0.7)

plt.tight_layout()

sup.plots.save_plot(path_save_figuras = path_save_figures,
                    file_name = 'histograma_2D_mapa',
                    formato = 'png',
                    dpi_ = 300)

plt.show()

## Filtro 50 km

Se aplicará un filtro por profundidad para diferenciar ambos catálogos a partir de los 50 km.

In [ ]:
# Diferenciar entre eventos menores a los 50 km y mayores a 50 km
profundidad_interes = 50 # km
filtro_prof_mayor = df_filtrado['Profundidad [km]'] > profundidad_interes
filtro_prof_menor_igual = df_filtrado['Profundidad [km]'] <= profundidad_interes


## Histograma Distancia

In [ ]:
step_ = 5 # km
dist_min = 0
dist_max = step_ * (1 + int(df_filtrado['Distancia [km]'].max() // step_) )

new_bins = np.arange(start=dist_min, stop=dist_max+step_, step=step_)

new_bins

In [ ]:
p10 = df_filtrado['Distancia [km]'].quantile(q=0.1).round(2)
p25 = df_filtrado['Distancia [km]'].quantile(q=0.25).round(2) # q1
p50 = df_filtrado['Distancia [km]'].quantile(q=0.5).round(2) # q2
p75 = df_filtrado['Distancia [km]'].quantile(q=0.75).round(2) # q3
p90 = df_filtrado['Distancia [km]'].quantile(q=0.9).round(2)

fig, ax = plt.subplots(figsize=(12,6))

sns.histplot(data = df_filtrado[filtro_prof_mayor],
             x = 'Distancia [km]',
             #log_scale = (False, True),
             bins = new_bins,
             ax= ax,
             legend= True,
             label= f'Depth $\geq{profundidad_interes}$ km',
             element='step',
             fill=False)

sns.histplot(data = df_filtrado[filtro_prof_menor_igual],
             x = 'Distancia [km]',
             #log_scale = (False, True),
             bins = new_bins,
             ax= ax,
             legend= True,
             label= f'Depth $<{profundidad_interes}$ km',
             element='step',
             fill=False)

# quantiles
ax.axvline(x=p10, label=f'P10: {p10} km', c='k', ls='--')
ax.axvline(x=p25, label=f'Q1: {p25} km', c='b', ls='--')
ax.axvline(x=p50, label=f'Q2: {p50} km', c='r', ls='--')
ax.axvline(x=p75, label=f'Q3: {p75} km', c='b', ls='--')
ax.axvline(x=p90, label=f'P90: {p90} km', c='k', ls='--')


ax.set_xlim(left=dist_min-step_, right=dist_max+step_)
ax.set_xticks(np.arange(start=dist_min, stop=dist_max+step_, step=step_*4))
ax.set_xlabel('Distance [km]')
ax.set_ylabel('Number of Events')
ax.grid(ls='--', c='gray', alpha=0.7)
ax.set_yscale('log')
ax.set_title('Distance to Campo Colorado')
ax.legend()

plt.tight_layout()

sup.plots.save_plot(path_save_figuras = path_save_figures,
                    file_name = 'hitograma_distancia_campo_colorado',
                    formato = 'png',
                    dpi_ = 300)

plt.show()

## Centrado al (0,0)

Se restará el punto de referencia a las coordenadas de todos los eventos, de modo que el punto central quede en (0, 0).

In [ ]:
df_filtrado['X [m] centrado'] = df_filtrado['X [m]'] - punto_xy_CampoColorado[0]
df_filtrado['Y [m] centrado'] = df_filtrado['Y [m]'] - punto_xy_CampoColorado[1]

df_filtrado.head()

## Histograma 2D centrado

In [ ]:
x_ticks_ = np.arange(start= (50 * np.floor(x_min/50)),
                     stop= (50 * np.floor(x_max/50)) + 50,
                     step=50)

x_ticks_

In [ ]:
x_min = df_filtrado['X [m] centrado'].min() / 1000
x_max = df_filtrado['X [m] centrado'].max() / 1000
y_min = df_filtrado['Y [m] centrado'].min() / 1000
y_max = df_filtrado['Y [m] centrado'].max() / 1000

x_ticks_ = np.arange(start= (50 * np.floor(x_min/50)),
                     stop= (50 * np.floor(x_max/50)) + 50,
                     step=50)

y_ticks_ = np.arange(start= (50 * np.floor(y_min/50)),
                     stop= (50 * np.floor(y_max/50)) + 50,
                     step=50)

steps_ = 5 # km
bins_x = np.arange(start=int(x_min)-steps_, stop=int(x_max)+steps_, step=steps_)
bins_y = np.arange(start=int(y_min)-steps_, stop=int(y_max)+steps_, step=steps_)

x_min, x_max = bins_x[0], bins_x[-1]
y_min, y_max = bins_y[0], bins_y[-1]


fig, ax = plt.subplots(ncols=2, nrows=1, figsize=(9,5))

### Axis 0
# Datos y bins (usa los tuyos: bins_x, bins_y)
x = df_filtrado['X [m] centrado'][filtro_prof_menor_igual].to_numpy() / 1000
y = df_filtrado['Y [m] centrado'][filtro_prof_menor_igual].to_numpy() / 1000

# hist2d con conteos en log (evita log(0) con vmin/cmin=1)
H = ax[0].hist2d(
    x, y,
    bins=[bins_x, bins_y],
    cmap='plasma', # rampa naranja
    norm=LogNorm(vmin=1), # escala log (base indiferente; la etiqueta la ponemos base-10)
    cmin=1, # descarta bins con 0 para evitar log(0)
    shading='auto',
    alpha= 0.7
)

# Barra de color en LOG10 (ticks en potencias de 10)
cb = fig.colorbar(H[3], ax=ax[0], pad=0.01)
cb.locator = LogLocator(base=10) # ticks en 1, 10, 100, ...
cb.formatter = LogFormatter(base=10, labelOnlyBase=False)
cb.update_ticks()
cb.set_label('Events [log10]')

# Limites y overlays (usa los tuyos)
ax[0].set_xlim(x_min, x_max)
ax[0].set_ylim(y_min, y_max)
ax[0].set_title(f'Depth $\leq{profundidad_interes}$ km')

ax[0].scatter(x=0,
            y=0,
            label='Campo Colorado',
            c='r',
            s=20)

ax[0].legend()

ax[0].set_xlabel('Centered X [km]')
ax[0].set_ylabel('Centered Y [km]')
#ax.set_title('Histograma 2D (escala log10)')
ax[0].grid(ls='--', c='k', alpha=0.7)
ax[0].set_xticks(x_ticks_)
ax[0].set_yticks(y_ticks_)
ax[0].tick_params(axis='x', labelrotation=60)

### Axis 1
# Datos y bins (usa los tuyos: bins_x, bins_y)
x = df_filtrado['X [m] centrado'][filtro_prof_mayor].to_numpy() / 1000
y = df_filtrado['Y [m] centrado'][filtro_prof_mayor].to_numpy() / 1000

# hist2d con conteos en log (evita log(0) con vmin/cmin=1)
H = ax[1].hist2d(
    x, y,
    bins=[bins_x, bins_y],
    cmap='plasma', # rampa naranja
    norm=LogNorm(vmin=1), # escala log (base indiferente; la etiqueta la ponemos base-10)
    cmin=1, # descarta bins con 0 para evitar log(0)
    shading='auto',
    alpha= 0.7
)

# Barra de color en LOG10 (ticks en potencias de 10)
cb = fig.colorbar(H[3], ax=ax[1], pad=0.01)
cb.locator = LogLocator(base=10) # ticks en 1, 10, 100, ...
cb.formatter = LogFormatter(base=10, labelOnlyBase=False)
cb.update_ticks()
cb.set_label('Events [log10]')

# Limites y overlays (usa los tuyos)
ax[1].set_xlim(x_min, x_max)
ax[1].set_ylim(y_min, y_max)
ax[1].set_title(f'Depth $>{profundidad_interes}$ km')

ax[1].scatter(x=0,
            y=0,
            label='Campo Colorado',
            c='r',
            s=20)

ax[1].legend()

ax[1].set_xlabel('Centered X [km]')
ax[1].set_ylabel('Centered Y [km]')
#ax.set_title('Histograma 2D (escala log10)')
ax[1].grid(ls='--', c='k', alpha=0.7)
ax[1].set_xticks(x_ticks_)
ax[1].set_yticks(y_ticks_)
ax[1].tick_params(axis='x', labelrotation=60)


plt.tight_layout()

sup.plots.save_plot(path_save_figuras = path_save_figures,
                    file_name = 'histograma_2D_mapa_centrado',
                    formato = 'png',
                    dpi_ = 300)

plt.show()

In [ ]:
x_min = df_filtrado['X [m] centrado'].min() / 1000
x_max = df_filtrado['X [m] centrado'].max() / 1000
y_min = df_filtrado['Y [m] centrado'].min() / 1000
y_max = df_filtrado['Y [m] centrado'].max() / 1000

x_ticks_ = np.arange(start= (10 * np.floor(x_min/10)),
                     stop= (10 * np.floor(x_max/10)) + 10,
                     step=10)

y_ticks_ = np.arange(start= (10 * np.floor(y_min/10)),
                     stop= (10 * np.floor(y_max/10)) + 10,
                     step=10)

steps_ = 2.5 # km
bins_x = np.arange(start=int(x_min)-steps_, stop=int(x_max)+steps_, step=steps_)
bins_y = np.arange(start=int(y_min)-steps_, stop=int(y_max)+steps_, step=steps_)

x_min, x_max = -100, 100 #bins_x[0], bins_x[-1]
y_min, y_max = -150, 150 #bins_y[0], bins_y[-1]


fig, ax = plt.subplots(ncols=2, nrows=1, figsize=(9,5))

### Axis 0
# Datos y bins (usa los tuyos: bins_x, bins_y)
x = df_filtrado['X [m] centrado'][filtro_prof_menor_igual].to_numpy() / 1000
y = df_filtrado['Y [m] centrado'][filtro_prof_menor_igual].to_numpy() / 1000

# hist2d con conteos en log (evita log(0) con vmin/cmin=1)
H = ax[0].hist2d(
    x, y,
    bins=[bins_x, bins_y],
    cmap='plasma', # rampa naranja
    norm=LogNorm(vmin=1), # escala log (base indiferente; la etiqueta la ponemos base-10)
    cmin=1, # descarta bins con 0 para evitar log(0)
    shading='auto',
    alpha= 0.7
)

# Barra de color en LOG10 (ticks en potencias de 10)
cb = fig.colorbar(H[3], ax=ax[0], pad=0.01)
cb.locator = LogLocator(base=10) # ticks en 1, 10, 100, ...
cb.formatter = LogFormatter(base=10, labelOnlyBase=False)
cb.update_ticks()
cb.set_label('Events [log10]')

ax[0].set_title(f'Depth $\leq{profundidad_interes}$ km')

ax[0].scatter(x=0,
            y=0,
            label='Campo Colorado',
            c='r',
            s=20)

ax[0].legend()

ax[0].set_xlabel('Centered X [km]')
ax[0].set_ylabel('Centered Y [km]')
#ax.set_title('Histograma 2D (escala log10)')
ax[0].grid(ls='--', c='k', alpha=0.7)
ax[0].set_xticks(x_ticks_)
ax[0].set_yticks(y_ticks_)
ax[0].tick_params(axis='x', labelrotation=60)

### Axis 1
# Datos y bins (usa los tuyos: bins_x, bins_y)
x = df_filtrado['X [m] centrado'][filtro_prof_mayor].to_numpy() / 1000
y = df_filtrado['Y [m] centrado'][filtro_prof_mayor].to_numpy() / 1000

# hist2d con conteos en log (evita log(0) con vmin/cmin=1)
H = ax[1].hist2d(
    x, y,
    bins=[bins_x, bins_y],
    cmap='plasma', # rampa naranja
    norm=LogNorm(vmin=1), # escala log (base indiferente; la etiqueta la ponemos base-10)
    cmin=1, # descarta bins con 0 para evitar log(0)
    shading='auto',
    alpha= 0.7
)

# Barra de color en LOG10 (ticks en potencias de 10)
cb = fig.colorbar(H[3], ax=ax[1], pad=0.01)
cb.locator = LogLocator(base=10) # ticks en 1, 10, 100, ...
cb.formatter = LogFormatter(base=10, labelOnlyBase=False)
cb.update_ticks()
cb.set_label('Events [log10]')

ax[1].set_title(f'Depth $>{profundidad_interes}$ km')

ax[1].scatter(x=0,
            y=0,
            label='Campo Colorado',
            c='r',
            s=20)

ax[1].legend()

ax[1].set_xlabel('Centered X [km]')
ax[1].set_ylabel('Centered Y [km]')
#ax.set_title('Histograma 2D (escala log10)')
ax[1].grid(ls='--', c='k', alpha=0.7)
ax[1].set_xticks(x_ticks_)
ax[1].set_yticks(y_ticks_)
ax[1].tick_params(axis='x', labelrotation=60)

# Limites y overlays (usa los tuyos)
ax[0].set_xlim(x_min, x_max)
ax[0].set_ylim(y_min, y_max)

ax[1].set_xlim(x_min, x_max)
ax[1].set_ylim(y_min, y_max)

plt.tight_layout()

sup.plots.save_plot(path_save_figuras = path_save_figures,
                    file_name = 'histograma_2D_mapa_centrado_zoom',
                    formato = 'png',
                    dpi_ = 300)

plt.show()

# Fin